In [0]:
# # %pip install torch==2.1.2 femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2

%pip install femr==0.2.3 xformers transformers==4.35.2
dbutils.library.restartPython()

In [0]:
# %pip install --force-reinstall --no-cache-dir torch==2.10.0 xformers==0.0.34 femr==0.2.3 transformers==4.35.2 datasets==2.15.0
# dbutils.library.restartPython()

In [0]:
%pip install --force-reinstall --no-cache-dir torch==2.1.2 femr==0.2.3 datasets==2.15.0 xformers==0.0.23.post1 transformers==4.35.2
dbutils.library.restartPython()

In [0]:
import sys
print([p for p in sys.path if 'site-packages' in p])

In [0]:
%pip install --force-reinstall --no-cache-dir "urllib3<2"
dbutils.library.restartPython()

In [0]:
from huggingface_hub import login

hf_token = dbutils.secrets.get(scope="hf-secrets", key="hf_token")
login(token=hf_token)

In [0]:
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime
import numpy as np

model_name = "StanfordShahLab/clmbr-t-base"

# Load tokenizer / batch loader
tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)

model = femr.models.transformer.FEMRModel.from_pretrained(model_name)

# Create an example patient to run inference on
# This patient follows the MEDS schema: https://github.com/Medical-Event-Data-Standard
example_patient = {
    'patient_id': 30,
    'events': [{
        'time': datetime.datetime(2011, 5, 8),
        'measurements': [
            {'code': 'SNOMED/184099003'},
            {'code': 'Visit/IP'},
        ],
    },
    {
        'time': datetime.datetime(2012, 6, 9),
        'measurements': [
            {'code': 'Visit/OP'},
            {'code': 'SNOMED/3950001'}
        ],
    }]
}

raw_batch = batch_processor.convert_patient(example_patient, tensor_type="pt")
batch = batch_processor.collate([raw_batch])

# Run model
with torch.no_grad():
    _, result = model(**batch)
    print(result['timestamps'].cpu().numpy().astype('datetime64[s]'))
    print(result['patient_ids'])
    print(result['representations'])


print(result.keys())
print(result['representations'].shape)

In [0]:
import sys
sys.modules['flash_attn'] = None
sys.modules['flash_attn_2_cuda'] = None

import femr.models.transformer
import femr.models.tokenizer
import femr.models.processor
import femr.models.processor as femr_processor_module
import femr.models.xformers as xformers_module
import torch
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.compute as pc
import datasets
import gc
import time
from itertools import groupby
from operator import itemgetter

# ============================================================================
# Config
# ============================================================================
# INPUT
model_name = "StanfordShahLab/clmbr-t-base"
meds_path = "/path/to/exports/meds_output_spark/data"

#OUTPUT - CHANGE WITH EACH ITERATION!!!!
output_table = "catalog.schema.patient_clmbr_embeddings_new_v5"
excluded_table = "catalog.schema.patient_clmbr_embeddings_excluded_new_v5"

SAMPLE_SIZE = None
WRITE_CHUNK_SIZE = 5000
MAX_SEQ_LEN = None

if not torch.cuda.is_available():
    raise RuntimeError("No GPU available -- stopping rather than silently running on CPU.")
device = "cuda"
print(f"torch={torch.__version__}, cuda={torch.version.cuda}, GPU={torch.cuda.get_device_name(0)}")

if SAMPLE_SIZE is not None:
    output_table = output_table + "_sample"
    excluded_table = excluded_table + "_sample"
    print(f"SAMPLE_SIZE={SAMPLE_SIZE} -- writing to {output_table} / {excluded_table}")


# ----------------------------------------------------------------------------
# Batching helpers
# ----------------------------------------------------------------------------
def make_batch(batch_processor, patients, timestamps_per_patient, tensor_type="pt", max_seq_len=None):
    creator = batch_processor.creator
    creator.start_batch()
    for p in patients:
        if max_seq_len is not None:
            n_events = timestamps_per_patient[p['patient_id']]
            if n_events > max_seq_len:
                creator.add_patient(p, offset=n_events - max_seq_len, max_length=max_seq_len)
                continue
        creator.add_patient(p)
    batch_data = creator.get_batch_data()
    if tensor_type is not None:
        formatter = datasets.formatting.get_formatter(tensor_type)
        batch_data = formatter.recursive_tensorize(batch_data)
    batch_data = creator.cleanup_batch(batch_data)
    return {"batch": femr_processor_module._add_dimension(batch_data)}


def move_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    else:
        return obj


def get_batch_size(seq_len, target_tokens=4096, max_batch_size=64):
    return max(1, min(max_batch_size, target_tokens // max(seq_len, 1)))


# ----------------------------------------------------------------------------
# Verify real attn_bias on the live forward pass -- idempotent across
# reruns in the same kernel session (fixed the recursion bug from earlier).
# ----------------------------------------------------------------------------
if not hasattr(xformers_module, '_true_original_wrapper'):
    xformers_module._true_original_wrapper = xformers_module.memory_efficient_attention_wrapper

_true_original_wrapper = xformers_module._true_original_wrapper
_checked = {'done': False}

def _wrapper_with_check(q, k, v, attn_bias):
    if not _checked['done']:
        assert q.device.type == "cuda", f"Expected cuda tensor, got {q.device.type}"
        assert attn_bias is not None, "attn_bias is None -- patch is still active somewhere!"
        print(f"VERIFIED: live forward pass using real xformers CUDA kernel with "
              f"real {type(attn_bias).__name__} (not CPU fallback, not disabled masking).")
        _checked['done'] = True
    return _true_original_wrapper(q, k, v, attn_bias)

xformers_module.memory_efficient_attention_wrapper = _wrapper_with_check


# ----------------------------------------------------------------------------
# Step 1: figure out the usable cohort (cheap columns only)
# ----------------------------------------------------------------------------
dataset = ds.dataset(meds_path, format="parquet")
id_table_df = dataset.to_table(columns=['subject_id', 'table']).to_pandas()
table_composition = id_table_df.groupby('subject_id')['table'].apply(set)
no_clinical_codes_ids = set(table_composition[table_composition.apply(lambda t: t == {'person'})].index)
usable_ids = sorted(set(id_table_df['subject_id'].unique()) - no_clinical_codes_ids)
del id_table_df

print(f"Total usable patients: {len(usable_ids)}")

if SAMPLE_SIZE is not None:
    usable_ids = usable_ids[:SAMPLE_SIZE]
    print(f"Sampled down to {len(usable_ids)} patients")

# ----------------------------------------------------------------------------
# Step 2: resume support -- skip patients already in the output table
# ----------------------------------------------------------------------------
already_done_ids = set()
try:
    existing = spark.table(output_table).select('patient_id').toPandas()
    already_done_ids = set(existing['patient_id'].tolist())
    print(f"Found {len(already_done_ids)} patients already in {output_table} -- will skip those.")
except Exception:
    print(f"No existing {output_table} found -- starting fresh.")

usable_ids = [pid for pid in usable_ids if pid not in already_done_ids]
print(f"{len(usable_ids)} patients remaining to process.")

# ----------------------------------------------------------------------------
# Step 3: load model once for the whole run (GPU memory proved bounded --
# no need to reload per chunk the way the CPU pipeline's fresh-process
# approach required).
# ----------------------------------------------------------------------------
tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)
model = femr.models.transformer.FEMRModel.from_pretrained(model_name)
model.eval()
model.to(device)

torch.cuda.reset_peak_memory_stats()
print(f"Model loaded. allocated={torch.cuda.memory_allocated()/1e6:.1f} MB, "
      f"reserved={torch.cuda.memory_reserved()/1e6:.1f} MB")

# ----------------------------------------------------------------------------
# Step 4: process in chunks of WRITE_CHUNK_SIZE patients -- load just that
# chunk's rows, run inference, write to Delta (append), then discard
# everything and move to the next chunk.
# ----------------------------------------------------------------------------
t_overall_start = time.time()
total_written = 0
n_chunks = (len(usable_ids) + WRITE_CHUNK_SIZE - 1) // WRITE_CHUNK_SIZE

for chunk_idx in range(n_chunks):
    chunk_ids = usable_ids[chunk_idx * WRITE_CHUNK_SIZE: (chunk_idx + 1) * WRITE_CHUNK_SIZE]
    t_chunk_start = time.time()

    # ---- load just this chunk's rows ----
    filter_expr = pc.field('subject_id').isin(chunk_ids)
    df_chunk = dataset.to_table(columns=['subject_id', 'time', 'code'], filter=filter_expr).to_pandas()
    df_chunk = df_chunk.sort_values(['subject_id', 'time'])

    sub_list = df_chunk['subject_id'].to_numpy().tolist()
    time_list = df_chunk['time'].to_numpy().astype('datetime64[us]').tolist()
    code_list = df_chunk['code'].to_numpy().tolist()
    del df_chunk

    # ---- build patient dicts ----
    patients = []
    timestamps_per_patient = {}
    for subject_id, patient_rows in groupby(zip(sub_list, time_list, code_list), key=itemgetter(0)):
        events = []
        for t, time_rows in groupby(patient_rows, key=itemgetter(1)):
            measurements = [{'code': c} for _, _, c in time_rows]
            events.append({'time': t, 'measurements': measurements})
        patients.append({'patient_id': subject_id, 'events': events})
        timestamps_per_patient[subject_id] = len(events)

    patients_sorted = sorted(patients, key=lambda p: timestamps_per_patient[p['patient_id']])
    del patients

    # ---- run inference on this chunk ----
    all_patient_ids, all_embeddings, all_seq_lengths = [], [], []

    i = 0
    n = len(patients_sorted)
    while i < n:
        probe_len = timestamps_per_patient[patients_sorted[min(i + 15, n - 1)]['patient_id']]
        bs = get_batch_size(probe_len)
        batch_patients = patients_sorted[i:i + bs]

        batch = make_batch(batch_processor, batch_patients, timestamps_per_patient,
                            tensor_type="pt", max_seq_len=MAX_SEQ_LEN)
        batch = move_to_device(batch, device)

        with torch.no_grad():
            _, result = model(**batch)

        pids = result['patient_ids'].cpu().numpy()
        times = result['timestamps'].cpu().numpy().astype('datetime64[s]')
        reps = result['representations'].cpu().numpy()

        res_df = pd.DataFrame({'patient_id': pids, 'time': times, 'row': np.arange(len(pids))})
        last_rows = res_df.loc[res_df.groupby('patient_id')['time'].idxmax()]

        for pid, row in zip(last_rows['patient_id'], last_rows['row']):
            all_patient_ids.append(pid)
            all_embeddings.append(reps[row])
            all_seq_lengths.append(timestamps_per_patient[pid])

        i += len(batch_patients)

    # ---- write this chunk to Delta (append) ----
    embeddings_array = np.stack(all_embeddings)
    emb_df = pd.DataFrame(embeddings_array, columns=[f'emb_{j}' for j in range(embeddings_array.shape[1])])
    emb_df.insert(0, 'patient_id', all_patient_ids)
    emb_df['sequence_length'] = all_seq_lengths

    spark.createDataFrame(emb_df).write.format("delta").mode("append").saveAsTable(output_table)
    total_written += len(emb_df)

    chunk_time = time.time() - t_chunk_start
    overall_rate = total_written / (time.time() - t_overall_start)
    print(f"Chunk {chunk_idx + 1}/{n_chunks}: wrote {len(emb_df)} embeddings "
          f"({chunk_time:.1f}s, {overall_rate:.1f} patients/sec overall, "
          f"{total_written}/{len(usable_ids)} total)")

    # ---- discard everything from this chunk ----
    del patients_sorted, all_patient_ids, all_embeddings, all_seq_lengths, emb_df, embeddings_array
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nDone: {total_written} embeddings written to {output_table} "
      f"({time.time() - t_overall_start:.1f}s total)")

# ----------------------------------------------------------------------------
# Step 5: excluded-patients audit table (once, at the end)
# ----------------------------------------------------------------------------
excluded_df = pd.DataFrame({
    'patient_id': list(no_clinical_codes_ids),
    'exclusion_reason': 'no_usable_clinical_codes'
})
spark.createDataFrame(excluded_df).write.format("delta").mode("overwrite").saveAsTable(excluded_table)
print(f"Saved {len(excluded_df)} excluded patient IDs to {excluded_table}")

In [0]:
import sys
sys.modules['flash_attn'] = None
sys.modules['flash_attn_2_cuda'] = None

import femr.models.transformer
import femr.models.tokenizer
import femr.models.processor
import femr.models.processor as femr_processor_module
import femr.models.xformers as xformers_module
import torch
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.compute as pc
import datasets
import gc
import time
from itertools import groupby
from operator import itemgetter

# ============================================================================
# Config -- all hyperparameters centralized here
# ============================================================================
CFG = {
    "model_name": "StanfordShahLab/clmbr-t-base",
    "meds_path": "/path/to/exports/meds_output_spark/data",
    "cohort_path": "catalog.schema.step7_filtered_cohort_with_apos",

    "output_table_month": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_month",
    "output_table_trimester": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester",
    "excluded_table": "catalog.schema.patient_clmbr_embeddings_excluded_new_v5_windowed",

    "sample_size": None,
    "write_chunk_size": 5000,
    "max_seq_len": None,
    "embed_dim": 768,  # clmbr-t-base hidden size

    "month_roll_pre_weeks": 26,
    "month_roll_post_weeks": 48,
    "month_roll_stride_weeks": 1,
    "month_roll_window_months": 1,

    "tri_roll_pre_months": 6,
    "tri_roll_post_months": 13,
    "tri_roll_stride_months": 1,
    "tri_roll_window_months": 3,
}

if CFG["sample_size"] is not None:
    suffix = "_sample"
    CFG["output_table_month"] += suffix
    CFG["output_table_trimester"] += suffix
    CFG["excluded_table"] += suffix
    print(f"sample_size={CFG['sample_size']} -- writing to *_sample tables")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU available -- stopping rather than silently running on CPU.")
device = "cuda"
print(f"torch={torch.__version__}, cuda={torch.version.cuda}, GPU={torch.cuda.get_device_name(0)}")


# ============================================================================
# Window boundary construction -- VECTORIZED across the whole cohort
# ============================================================================
def precompute_rolling_month_boundaries(starts, pre_weeks, post_weeks, stride_weeks, window_months):
    names = []
    wstart_cols, wend_cols = [], []
    offsets = range(-pre_weeks, post_weeks, stride_weeks)
    for offset in offsets:
        names.append(f"month_wk{offset:+03d}")
        wstart = starts + pd.Timedelta(weeks=offset)
        wend = wstart + pd.DateOffset(months=window_months)
        wstart_cols.append(wstart.values)
        wend_cols.append(wend.values)
    return names, np.column_stack(wstart_cols), np.column_stack(wend_cols)


def precompute_rolling_trimester_boundaries(starts, pre_months, post_months, stride_months, window_months):
    names = []
    wstart_cols, wend_cols = [], []
    offsets = range(-pre_months, post_months, stride_months)
    for offset in offsets:
        names.append(f"tri_mo{offset:+03d}")
        wstart = starts + pd.DateOffset(months=offset)
        wend = wstart + pd.DateOffset(months=window_months)
        wstart_cols.append(wstart.values)
        wend_cols.append(wend.values)
    return names, np.column_stack(wstart_cols), np.column_stack(wend_cols)


def extract_windows_vectorized(times_sorted, reps_sorted, wstarts, wends, embed_dim):
    n_windows = len(wstarts)
    if len(times_sorted) == 0:
        emb_matrix = np.full((n_windows, embed_dim), np.nan, dtype=np.float32)
        n_tokens = np.full(n_windows, np.nan)
        unavailable = np.ones(n_windows, dtype=np.int64)
        empty = np.zeros(n_windows, dtype=np.int64)
        return emb_matrix, n_tokens, unavailable, empty

    last_event_time = times_sorted[-1]
    first_event_time = times_sorted[0]

    wstarts = wstarts.astype(times_sorted.dtype)
    wends   = wends.astype(times_sorted.dtype)

    after_cutoff   = wends > last_event_time
    before_history = wends < first_event_time
    available      = ~after_cutoff & ~before_history

    idx = np.searchsorted(times_sorted, wends, side="right") - 1
    idx_clipped = np.clip(idx, 0, len(times_sorted) - 1)
    emb_matrix = reps_sorted[idx_clipped].astype(np.float32, copy=True)
    emb_matrix[~available] = np.nan

    lo = np.searchsorted(times_sorted, wstarts, side="left")
    hi = np.searchsorted(times_sorted, wends, side="left")
    n_tokens = (hi - lo).astype(np.float64)
    n_tokens[after_cutoff] = np.nan  # window incomplete -- true count unknown

    unavailable = (~available).astype(np.int64)
    empty = ((n_tokens == 0) & available).astype(np.int64)

    return emb_matrix, n_tokens, unavailable, empty


def build_long_df_from_arrays(pids_repeated, names_tiled, emb_matrix, n_tokens, unavailable, empty):
    """Assembles the final long-format DataFrame from already-vectorized
    arrays -- a single DataFrame construction, no per-row Python loop."""
    df = pd.DataFrame(emb_matrix, columns=[f"emb_{d}" for d in range(emb_matrix.shape[1])])
    df.insert(0, "window_name", names_tiled)
    df.insert(0, "patient_id", pids_repeated)
    df["n_tokens"]    = n_tokens
    df["unavailable"] = unavailable
    df["empty"]       = empty
    return df


# ----------------------------------------------------------------------------
# Batching helpers 
# ----------------------------------------------------------------------------
def make_batch(batch_processor, patients, timestamps_per_patient, tensor_type="pt", max_seq_len=None):
    creator = batch_processor.creator
    creator.start_batch()
    for p in patients:
        if max_seq_len is not None:
            n_events = timestamps_per_patient[p['patient_id']]
            if n_events > max_seq_len:
                creator.add_patient(p, offset=n_events - max_seq_len, max_length=max_seq_len)
                continue
        creator.add_patient(p)
    batch_data = creator.get_batch_data()
    if tensor_type is not None:
        formatter = datasets.formatting.get_formatter(tensor_type)
        batch_data = formatter.recursive_tensorize(batch_data)
    batch_data = creator.cleanup_batch(batch_data)
    return {"batch": femr_processor_module._add_dimension(batch_data)}


def move_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    else:
        return obj


def get_batch_size(seq_len, target_tokens=4096, max_batch_size=64):
    return max(1, min(max_batch_size, target_tokens // max(seq_len, 1)))


# ----------------------------------------------------------------------------
# xformers attn_bias verification
# ----------------------------------------------------------------------------
if not hasattr(xformers_module, '_true_original_wrapper'):
    xformers_module._true_original_wrapper = xformers_module.memory_efficient_attention_wrapper

_true_original_wrapper = xformers_module._true_original_wrapper
_checked = {'done': False}

def _wrapper_with_check(q, k, v, attn_bias):
    if not _checked['done']:
        assert q.device.type == "cuda", f"Expected cuda tensor, got {q.device.type}"
        assert attn_bias is not None, "attn_bias is None -- patch is still active somewhere!"
        print(f"VERIFIED: live forward pass using real xformers CUDA kernel with "
              f"real {type(attn_bias).__name__} (not CPU fallback, not disabled masking).")
        _checked['done'] = True
    return _true_original_wrapper(q, k, v, attn_bias)

xformers_module.memory_efficient_attention_wrapper = _wrapper_with_check


# ----------------------------------------------------------------------------
# Step 1: cohort
# ----------------------------------------------------------------------------
t_setup_start = time.time()

cohort_df = spark.table(CFG["cohort_path"]).select(
    "person_id", "recorded_episode_start", "inferred_episode_start"
).toPandas()
print(f"[timing] cohort spark.table().toPandas(): {time.time()-t_setup_start:.1f}s")

cohort_df["recorded_episode_start"] = pd.to_datetime(cohort_df["recorded_episode_start"])
cohort_df["inferred_episode_start"] = pd.to_datetime(cohort_df["inferred_episode_start"])

cohort_df["episode_start"] = cohort_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)

n_no_start = cohort_df["episode_start"].isna().sum()
if n_no_start > 0:
    print(f"WARNING: {n_no_start} patients have no episode_start from either source -- "
          f"these will be excluded (cannot anchor windows without a start date).")
cohort_df = cohort_df.dropna(subset=["episode_start"]).reset_index(drop=True)

episode_lookup = {row.person_id: row.episode_start for row in cohort_df.itertuples()}
print(f"Episode start resolved for {len(episode_lookup)} patients.")

# ---- precompute ALL window boundaries for the whole cohort ONCE, vectorized ----
t0 = time.time()
MONTH_NAMES, MONTH_WSTART_ALL, MONTH_WEND_ALL = precompute_rolling_month_boundaries(
    cohort_df["episode_start"],
    pre_weeks=CFG["month_roll_pre_weeks"],
    post_weeks=CFG["month_roll_post_weeks"],
    stride_weeks=CFG["month_roll_stride_weeks"],
    window_months=CFG["month_roll_window_months"],
)
TRI_NAMES, TRI_WSTART_ALL, TRI_WEND_ALL = precompute_rolling_trimester_boundaries(
    cohort_df["episode_start"],
    pre_months=CFG["tri_roll_pre_months"],
    post_months=CFG["tri_roll_post_months"],
    stride_months=CFG["tri_roll_stride_months"],
    window_months=CFG["tri_roll_window_months"],
)
pid_to_row = {pid: i for i, pid in enumerate(cohort_df["person_id"].values)}
print(f"[timing] vectorized boundary precompute for {len(pid_to_row)} patients "
      f"({len(MONTH_NAMES)} rolling-month + {len(TRI_NAMES)} rolling-trimester windows): "
      f"{time.time()-t0:.2f}s")

# ----------------------------------------------------------------------------
# Step 2: figure out the usable cohort from MEDS 
# ----------------------------------------------------------------------------
dataset = ds.dataset(CFG["meds_path"], format="parquet")
t0 = time.time()
id_table_df = dataset.to_table(columns=['subject_id', 'table']).to_pandas()
print(f"[timing] full-dataset id/table scan (runs regardless of sample_size): {time.time()-t0:.1f}s")
table_composition = id_table_df.groupby('subject_id')['table'].apply(set)
no_clinical_codes_ids = set(table_composition[table_composition.apply(lambda t: t == {'person'})].index)
usable_ids = sorted(set(id_table_df['subject_id'].unique()) - no_clinical_codes_ids)
del id_table_df

usable_ids = [pid for pid in usable_ids if pid in episode_lookup]
print(f"Total usable patients with clinical codes + resolvable episode: {len(usable_ids)}")

if CFG["sample_size"] is not None:
    usable_ids = usable_ids[:CFG["sample_size"]]
    print(f"Sampled down to {len(usable_ids)} patients")

# ----------------------------------------------------------------------------
# Step 3: resume support -- skip patients already in the month-level output table
# ----------------------------------------------------------------------------
already_done_ids = set()
try:
    existing = spark.table(CFG["output_table_month"]).select('patient_id').distinct().toPandas()
    already_done_ids = set(existing['patient_id'].tolist())
    print(f"Found {len(already_done_ids)} patients already in {CFG['output_table_month']} -- will skip those.")
except Exception:
    print(f"No existing {CFG['output_table_month']} found -- starting fresh.")

usable_ids = [pid for pid in usable_ids if pid not in already_done_ids]
print(f"{len(usable_ids)} patients remaining to process.")

# ----------------------------------------------------------------------------
# Step 4: load model once for the whole run
# ----------------------------------------------------------------------------
tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(CFG["model_name"])
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)
t0 = time.time()
model = femr.models.transformer.FEMRModel.from_pretrained(CFG["model_name"])
print(f"[timing] model download/load from_pretrained: {time.time()-t0:.1f}s")
model.eval()
model.to(device)

torch.cuda.reset_peak_memory_stats()
print(f"Model loaded. allocated={torch.cuda.memory_allocated()/1e6:.1f} MB, "
      f"reserved={torch.cuda.memory_reserved()/1e6:.1f} MB")

# ----------------------------------------------------------------------------
# Step 5: process in chunks
# ----------------------------------------------------------------------------
t_overall_start = time.time()
total_written = 0
n_chunks = (len(usable_ids) + CFG["write_chunk_size"] - 1) // CFG["write_chunk_size"]

for chunk_idx in range(n_chunks):
    chunk_ids = usable_ids[chunk_idx * CFG["write_chunk_size"]: (chunk_idx + 1) * CFG["write_chunk_size"]]
    t_chunk_start = time.time()

    filter_expr = pc.field('subject_id').isin(chunk_ids)
    df_chunk = dataset.to_table(columns=['subject_id', 'time', 'code'], filter=filter_expr).to_pandas()
    df_chunk = df_chunk.sort_values(['subject_id', 'time'])

    sub_list  = df_chunk['subject_id'].to_numpy().tolist()
    time_list = df_chunk['time'].to_numpy().astype('datetime64[us]').tolist()
    code_list = df_chunk['code'].to_numpy().tolist()
    del df_chunk

    patients = []
    timestamps_per_patient = {}
    for subject_id, patient_rows in groupby(zip(sub_list, time_list, code_list), key=itemgetter(0)):
        events = []
        for t, time_rows in groupby(patient_rows, key=itemgetter(1)):
            measurements = [{'code': c} for _, _, c in time_rows]
            events.append({'time': t, 'measurements': measurements})
        patients.append({'patient_id': subject_id, 'events': events})
        timestamps_per_patient[subject_id] = len(events)

    patients_sorted = sorted(patients, key=lambda p: timestamps_per_patient[p['patient_id']])
    del patients

    # accumulate per-patient arrays across the whole chunk (across all
    # mini-batches) -- stacked once per granularity at chunk end.
    m_pids, m_embs, m_ntok, m_unavail, m_empty = [], [], [], [], []
    t_pids, t_embs, t_ntok, t_unavail, t_empty = [], [], [], [], []

    i = 0
    n = len(patients_sorted)
    while i < n:
        probe_len = timestamps_per_patient[patients_sorted[min(i + 15, n - 1)]['patient_id']]
        bs = get_batch_size(probe_len)
        batch_patients = patients_sorted[i:i + bs]

        batch = make_batch(batch_processor, batch_patients, timestamps_per_patient,
                           tensor_type="pt", max_seq_len=CFG["max_seq_len"])
        batch = move_to_device(batch, device)

        with torch.no_grad():
            _, result = model(**batch)

        pids  = result['patient_ids'].cpu().numpy()
        times = result['timestamps'].cpu().numpy().astype('datetime64[s]')
        reps  = result['representations'].cpu().numpy()

        # group this batch's flattened (patient, time, rep) rows by patient,
        # sort each patient's tokens by time, then extract all three
        # granularities' boundary embeddings from the same token sequence.
        order = np.lexsort((times, pids))
        pids_o, times_o, reps_o = pids[order], times[order], reps[order]

        boundaries = np.searchsorted(pids_o, np.unique(pids_o))
        boundaries = np.append(boundaries, len(pids_o))
        unique_pids = np.unique(pids_o)

        for j, pid in enumerate(unique_pids):
            s, e = boundaries[j], boundaries[j + 1]
            p_times = times_o[s:e]
            p_reps  = reps_o[s:e]

            row_idx = pid_to_row[pid]

            m_emb, m_nt, m_un, m_em = extract_windows_vectorized(
                p_times, p_reps, MONTH_WSTART_ALL[row_idx], MONTH_WEND_ALL[row_idx], CFG["embed_dim"])
            t_emb, t_nt, t_un, t_em = extract_windows_vectorized(
                p_times, p_reps, TRI_WSTART_ALL[row_idx], TRI_WEND_ALL[row_idx], CFG["embed_dim"])

            m_pids.append(pid); m_embs.append(m_emb); m_ntok.append(m_nt); m_unavail.append(m_un); m_empty.append(m_em)
            t_pids.append(pid); t_embs.append(t_emb); t_ntok.append(t_nt); t_unavail.append(t_un); t_empty.append(t_em)

        i += len(batch_patients)

    # ---- write all three granularities for this chunk (long format) ----
    for (pids_l, embs_l, ntok_l, unavail_l, empty_l), names, table_name, label in [
        ((m_pids, m_embs, m_ntok, m_unavail, m_empty), MONTH_NAMES, CFG["output_table_month"], "month (rolling)"),
        ((t_pids, t_embs, t_ntok, t_unavail, t_empty), TRI_NAMES,   CFG["output_table_trimester"], "trimester (rolling)"),
    ]:
        n_windows = len(names)
        pids_repeated = np.repeat(pids_l, n_windows)
        names_tiled   = np.tile(names, len(pids_l))
        emb_matrix    = np.vstack(embs_l)
        n_tokens      = np.concatenate(ntok_l)
        unavailable   = np.concatenate(unavail_l)
        empty         = np.concatenate(empty_l)

        out_df = build_long_df_from_arrays(pids_repeated, names_tiled, emb_matrix,
                                           n_tokens, unavailable, empty)
        spark.createDataFrame(out_df).write.format("delta").mode("append").saveAsTable(table_name)
        n_patients_this_chunk = len(pids_l)
        print(f"  wrote {len(out_df)} rows ({n_patients_this_chunk} patients x {n_windows} windows) "
              f"to {table_name} ({label})")

    total_written += n_patients_this_chunk
    chunk_time = time.time() - t_chunk_start
    overall_rate = total_written / (time.time() - t_overall_start)
    print(f"Chunk {chunk_idx + 1}/{n_chunks}: {chunk_time:.1f}s, "
          f"{overall_rate:.1f} patients/sec overall, "
          f"{total_written}/{len(usable_ids)} total")

    del patients_sorted
    del m_pids, m_embs, m_ntok, m_unavail, m_empty
    del t_pids, t_embs, t_ntok, t_unavail, t_empty
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nDone: {total_written} patients written across rolling month/trimester tables "
      f"({time.time() - t_overall_start:.1f}s total)")

# ----------------------------------------------------------------------------
# Step 6: excluded-patients audit table
# ----------------------------------------------------------------------------
excluded_df = pd.DataFrame({
    'patient_id': list(no_clinical_codes_ids),
    'exclusion_reason': 'no_usable_clinical_codes'
})
spark.createDataFrame(excluded_df).write.format("delta").mode("overwrite").saveAsTable(CFG["excluded_table"])
print(f"Saved {len(excluded_df)} excluded patient IDs to {CFG['excluded_table']}")